In [45]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge

In [46]:
df = pd.read_csv("../data/creditpulse_data.csv")
df.head()

,tx_regularity_score,bill_payment_rate,income_stability,has_savings_behaviour,merchant_diversity,avg_monthly_inflow,account_age_months,credit_score
0,51.108777,63.927747,14.364215,1.0,5.0,5.000000,4.0,44.711974
1,41.567659,35.357812,14.638878,0.0,5.0,5.000000,5.0,25.322477
2,53.584160,58.135071,11.349188,1.0,2.0,69.983046,37.0,45.469374
3,29.081685,51.118945,18.366255,0.0,4.0,13.154627,37.0,29.150549
4,73.910790,19.554846,40.204323,0.0,4.0,16.008478,26.0,35.963409


In [47]:
FEATURE_COLS = [
    "tx_regularity_score",
    "bill_payment_rate",
    "income_stability",
    "has_savings_behaviour",
    "merchant_diversity",
    "avg_monthly_inflow",
    "account_age_months",
]

X = df[FEATURE_COLS]
y = df["credit_score"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y range: {y.min():.2f} → {y.max():.2f}")

X shape: (2000, 7)
y shape: (2000,)
y range: 9.88 → 66.04


In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {X_train.shape[0]} rows")
print(f"Test  : {X_test.shape[0]} rows")

Train : 1600 rows
Test  : 400 rows


In [49]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  Ridge(alpha=1.0))
])

pipeline.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [50]:
y_pred = pipeline.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}   (avg points off — lower is better)")
print(f"RMSE : {rmse:.4f}  (penalises big errors — lower is better)")
print(f"R²   : {r2:.4f}   (1.0 = perfect — higher is better)")

MAE  : 0.0048   (avg points off — lower is better)
RMSE : 0.0060  (penalises big errors — lower is better)
R²   : 1.0000   (1.0 = perfect — higher is better)


In [51]:
# Tests
amaka = pd.DataFrame([{
    "tx_regularity_score":   85.0,   
    "bill_payment_rate":     88.0, 
    "income_stability":      72.0,   
    "has_savings_behaviour": 1.0,
    "merchant_diversity":    10.0,  
    "avg_monthly_inflow":    150.0,  
    "account_age_months":    30.0,   
}])

emeka = pd.DataFrame([{
    "tx_regularity_score":   32.0,
    "bill_payment_rate":     40.0,
    "income_stability":      28.0,
    "has_savings_behaviour": 0.0,
    "merchant_diversity":    2.0,
    "avg_monthly_inflow":    30.0,
    "account_age_months":    6.0,
}])

amaka_score = pipeline.predict(amaka)[0]
emeka_score = pipeline.predict(emeka)[0]

print(f"Amaka score : {amaka_score:.1f}")
print(f"Emeka score : {emeka_score:.1f}")
print(f"Gap         : {amaka_score - emeka_score:.1f} points")

Amaka score : 77.9
Emeka score : 25.7
Gap         : 52.2 points


In [52]:
# feature importance
model        = pipeline.named_steps["model"]
scaler       = pipeline.named_steps["scaler"]
coefficients = model.coef_

# Coefficients are in scaled space — convert to relative importance
importance = np.abs(coefficients)
importance = importance / importance.sum() * 100

feat_imp = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "coefficient": coefficients,
    "importance %": importance
}).sort_values("importance %", ascending=False)

print(feat_imp.to_string(index=False))

              feature  coefficient  importance %
has_savings_behaviour     4.637292     23.417449
    bill_payment_rate     4.556328     23.008598
  tx_regularity_score     4.331196     21.871724
     income_stability     3.479665     17.571654
   account_age_months     1.145139      5.782739
   merchant_diversity     1.083239      5.470152
   avg_monthly_inflow     0.569860      2.877684


In [53]:
os.makedirs("../models", exist_ok=True)

# Save full pipeline (scaler + model together)
joblib.dump(pipeline, "../models/creditpulse_model.pkl")

# Lock column order — API must send features in this exact order
with open("../models/feature_columns.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
    
print("Saved → models/creditpulse_model.pkl")
print("Saved → models/feature_columns.json")

Saved → models/creditpulse_model.pkl
Saved → models/feature_columns.json


In [54]:
# Save metrics for methodology slide
report = f"""CreditPulse Model Report
========================
Algorithm   : XGBRegressor
Features    : {len(FEATURE_COLS)}
Train rows  : {X_train.shape[0]}
Test rows   : {X_test.shape[0]}

Metrics (test set)
------------------
MAE         : {mae:.4f}
RMSE        : {rmse:.4f}
R²          : {r2:.4f}

Persona Sanity Check
--------------------
Amaka score : {amaka_score:.1f}
Emeka score : {emeka_score:.1f}
"""

with open("../models/model_report.txt", "w") as f:
    f.write(report)
print("Saved → models/model_report.txt")

Saved → models/model_report.txt
